# Wide-Column Databases — First Contact

Imagine a spreadsheet where each row can have completely different columns, rows are sorted by a key you choose, and the whole thing is designed to be split across hundreds of machines. Cassandra never has a single point of failure — every node is equal, there is no master. Writes are always fast because they go to a commit log first, then memory. Reads are fast only if your query matches the partition key — otherwise Cassandra has to scan every node, which is expensive and explicit (`ALLOW FILTERING`).

## What makes wide-column stores different

- **Partition key** — determines which node holds the data. Choose wrong and all your data lands on one node (hot partition). Every query must include it.
- **Clustering key** — determines sort order within a partition. This is your only free `ORDER BY`.
- **No joins, no cross-partition aggregations** — design your tables around your queries, not your entities.
- **When to use** — time-series, IoT sensor data, write-heavy workloads, data that naturally partitions by ID + time.

In [8]:
# Note: gevent reactor is already handled inside get_cassandra_session()
from pathlib import Path
import sys

for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_cassandra_session
import pandas as pd

session = get_cassandra_session()
print("Connected to Cassandra.")

rows = session.execute("SELECT keyspace_name FROM system_schema.keyspaces")
for row in rows:
    print(" ", row.keyspace_name)

Connected to Cassandra.
  system_auth
  system_schema
  telemetry
  system_distributed
  system
  system_traces


In [9]:
# Explore the telemetry keyspace schema
# partition_key = endpoint_id  →  which node holds this endpoint's data
# clustering   = metric_id, recorded_at  →  sort order within the partition
# regular      = metric_name, value, unit

rows = session.execute("""
    SELECT column_name, type, kind
    FROM system_schema.columns
    WHERE keyspace_name = 'telemetry'
    AND table_name = 'metrics'
""")
df = pd.DataFrame(list(rows))
# Show partition/clustering keys first for clarity
order = {'partition_key': 0, 'clustering': 1, 'regular': 2}
df['_sort'] = df['kind'].map(order)
df = df.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)
print("telemetry.metrics schema:")
print(df.to_string(index=False))

telemetry.metrics schema:
column_name      type          kind
endpoint_id      uuid partition_key
  metric_id      uuid    clustering
recorded_at timestamp    clustering
metric_name      text       regular
       unit      text       regular
      value    double       regular


## 5 queries — Cassandra CQL style

All queries must use the partition key (`endpoint_id`). Full table scans require `ALLOW FILTERING` — Cassandra forces you to be explicit about the cost. Acceptable for learning; avoid in production.

In [10]:
# Query 1 — total row count (ALLOW FILTERING for learning only)
# In production you'd maintain a counter table, not scan the whole cluster.
result = session.execute(
    "SELECT COUNT(*) FROM telemetry.metrics ALLOW FILTERING"
)
print(f"Total metrics rows: {result.one()[0]:,}")

Total metrics rows: 1,000,001


In [11]:
# Query 2 — fetch metrics for one endpoint (the correct Cassandra pattern)
# Pull a real endpoint_id from Postgres to use as the partition key.
from db_connections import get_postgres_conn

pg = get_postgres_conn()
cur = pg.cursor()
cur.execute("SELECT endpoint_id FROM telemetry.endpoints LIMIT 1")
import uuid
sample_id = uuid.UUID(str(cur.fetchone()[0]))
print(f"Querying metrics for endpoint: {sample_id}")

rows = session.execute(
    "SELECT endpoint_id, metric_name, value, recorded_at, unit "
    "FROM telemetry.metrics "
    "WHERE endpoint_id = %s "
    "LIMIT 10",
    (sample_id,)
)
df2 = pd.DataFrame(list(rows))
print(f"Rows returned: {len(df2)}")
print(df2.to_string(index=False))

Querying metrics for endpoint: 50fdd579-de84-4899-a774-a3021d2cfe36
Rows returned: 10
                         endpoint_id    metric_name   value             recorded_at    unit
50fdd579-de84-4899-a774-a3021d2cfe36    cpu_percent   72.50 2026-03-24 02:40:57.121 percent
50fdd579-de84-4899-a774-a3021d2cfe36 memory_percent   14.98 2026-03-21 19:26:39.063 percent
50fdd579-de84-4899-a774-a3021d2cfe36        disk_io  536.32 2026-03-20 16:09:21.063    iops
50fdd579-de84-4899-a774-a3021d2cfe36        disk_io 3262.29 2026-03-19 00:39:54.063    iops
50fdd579-de84-4899-a774-a3021d2cfe36     network_in  327.25 2026-03-18 09:32:44.063    mbps
50fdd579-de84-4899-a774-a3021d2cfe36        disk_io 3545.93 2026-03-16 11:09:55.063    iops
50fdd579-de84-4899-a774-a3021d2cfe36    network_out  837.90 2026-03-10 08:46:13.063    mbps
50fdd579-de84-4899-a774-a3021d2cfe36     network_in  816.31 2026-03-05 05:16:19.063    mbps
50fdd579-de84-4899-a774-a3021d2cfe36     network_in   38.01 2026-03-03 16:55:36.063   

In [12]:
# Query 3 — filter by metric_name within one partition
# metric_name is a regular column (not a clustering key) so ALLOW FILTERING
# is required — but the scan is scoped to ONE partition (one endpoint), not the whole cluster.
rows = session.execute(
    "SELECT metric_name, value, recorded_at "
    "FROM telemetry.metrics "
    "WHERE endpoint_id = %s "
    "AND metric_name = 'cpu_percent' "
    "LIMIT 5 ALLOW FILTERING",
    (sample_id,)
)
df3 = pd.DataFrame(list(rows))
print("CPU metrics for one endpoint (5 rows):")
print(df3.to_string(index=False))

CPU metrics for one endpoint (5 rows):
metric_name  value             recorded_at
cpu_percent  72.50 2026-03-24 02:40:57.121
cpu_percent  89.58 2026-02-27 20:50:56.063
cpu_percent  58.39 2026-02-03 09:02:12.063
cpu_percent  54.76 2026-01-29 08:03:54.063
cpu_percent  85.00 2026-01-12 10:03:57.063


In [13]:
# Query 4 — write a new metric row
# Cassandra writes go to the commit log + memtable first — always fast.
# No transaction, no lock, no rollback. That's the write-speed tradeoff.
import uuid, datetime

session.execute("""
    INSERT INTO telemetry.metrics
        (endpoint_id, metric_id, recorded_at, metric_name, value, unit)
    VALUES (%s, %s, %s, %s, %s, %s)
""", (
    sample_id,
    uuid.uuid4(),
    datetime.datetime.utcnow(),
    'cpu_percent',
    72.5,
    'percent',
))
print("Write successful — Cassandra commit log accepted it instantly.")
print("No transaction, no lock, no rollback — that's the tradeoff.")

Write successful — Cassandra commit log accepted it instantly.
No transaction, no lock, no rollback — that's the tradeoff.


C:\Users\shareuser\AppData\Local\Temp\ipykernel_16824\3933260781.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow(),


In [14]:
# Query 5 — demonstrate partition distribution
# Counts per endpoint should be roughly equal — that's good partition design.
# Wildly uneven counts = hot partition problem.
cur.execute("SELECT endpoint_id FROM telemetry.endpoints LIMIT 5")
sample_ids = [uuid.UUID(str(row[0])) for row in cur.fetchall()]

print("Row counts per endpoint (roughly equal = good partitioning):")
for eid in sample_ids:
    result = session.execute(
        "SELECT COUNT(*) FROM telemetry.metrics WHERE endpoint_id = %s",
        (eid,)
    )
    count = result.one()[0]
    bar = '#' * min(int(count // 5), 40)
    print(f"  {str(eid)[:8]}...: {count:>3} rows  {bar}")

Row counts per endpoint (roughly equal = good partitioning):
  50fdd579...:  46 rows  #########
  996614fd...:  26 rows  #####
  648305fe...:  39 rows  #######
  b8c887a2...:  45 rows  #########
  50133629...:  35 rows  #######


## SQL vs CQL — same question, different rules

| SQL (Postgres) | CQL (Cassandra) |
|---|---|
| `WHERE any_column = x` | `WHERE partition_key = x` (required) |
| `GROUP BY` | No aggregations across partitions |
| `ORDER BY any_column` | `ORDER BY clustering_key` only |
| `JOIN` | No joins — denormalise instead |
| `UPDATE any_row` | `INSERT` / `UPDATE` by full primary key only |
| `EXPLAIN ANALYZE` | `TRACING ON` — shows token routing across nodes |

## Key observations

- **Partition key is everything** — every query must include it or you pay the `ALLOW FILTERING` tax (full cluster scan across all nodes).
- **Write path is always fast** — data goes to the commit log + memtable first; disk flush happens later in the background via SSTable compaction. Reads only get this guarantee if you query by partition key.
- **Design for queries, not entities** — in relational you design normalised tables then write queries. In Cassandra you design tables *around* your known queries upfront. Changing access patterns later means creating new tables.
- **Schema above** — `endpoint_id` (partition key) + `metric_id` + `recorded_at` (clustering keys). Each endpoint's entire metric history lives in one partition, sorted by time. That's the textbook Cassandra pattern for time-series.
- **Citi hook** — telemetry metrics partitioned by `endpoint_id`, clustered by `recorded_at`, is exactly how a Citi infrastructure monitoring team would store 500K+ sensor readings: one partition per device, time-ordered within, no cross-device joins needed.